# Notebook 04: Battery energy storage footprint

In [1]:
SECTOR = "Battery"
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
from IPython.display import display

# I use the verified Notebook 03 exports without repeating the county spatial joins.
ROOT = Path(r"G:/My Drive/Projects/Large projects/2026TexasEnergyFootprint")
INPUT = ROOT / "05_Optimized_Data" / "GeoParquet"
OUTPUT = ROOT / "06_Energy_Footprint_Analysis" / SECTOR
OUTPUT.mkdir(parents=True, exist_ok=True)
ACRE_M2 = 4046.8564224
FT_PER_MILE = 5280.0
SQFT_PER_ACRE = 43560.0
SCENARIOS = ("low", "central", "high")

def read_layer(name):
    path = INPUT / f"{name}.parquet"
    if not path.is_file():
        raise FileNotFoundError(f"Missing Notebook 03 export: {path}")
    data = gpd.read_parquet(path)
    if data.crs is None:
        raise ValueError(f"{name}: missing CRS")
    if data.geometry.isna().any() or data.geometry.is_empty.any():
        raise ValueError(f"{name}: missing or empty geometries")
    return data

def county_key(df):
    if "county_geoid" not in df:
        raise KeyError("Notebook 03 county_geoid is required")
    s = df["county_geoid"].astype("string").str.replace(r"\.0$", "", regex=True).str.zfill(5)
    return s.where(s.str.fullmatch(r"48\d{3}"), pd.NA)

def save_results(detail, county, state, assumptions):
    # I save both geographic levels and verify that each scenario reconciles.
    for scenario in SCENARIOS:
        col = f"{scenario}_acres"
        if col not in detail or col not in county or col not in state:
            raise KeyError(f"Missing scenario output: {col}")
        a = float(detail[col].sum()); b = float(county[col].sum()); c = float(state[col].sum())
        if not np.isclose(a, c, rtol=1e-9, atol=0.01):
            raise AssertionError(f"Statewide {scenario} does not match facility detail")
        # I report unassigned county acreage separately rather than force a county allocation.
        unassigned = float(detail.loc[detail['county_geoid'].isna(), col].sum())
        if not np.isclose(b + unassigned, c, rtol=1e-9, atol=0.01):
            raise AssertionError(f"County plus unassigned acreage does not reconcile: {scenario}")
    detail.drop(columns=['geometry'], errors='ignore').to_csv(OUTPUT / 'facility_scenarios.csv', index=False)
    county.to_csv(OUTPUT / 'county_scenarios.csv', index=False)
    state.to_csv(OUTPUT / 'statewide_scenarios.csv', index=False)
    assumptions.to_csv(OUTPUT / 'scenario_assumptions.csv', index=False)
    print('Saved:', OUTPUT)
    display(state.round(2))
    display(county.head(10).round(2))


In [2]:
battery = read_layer("Texas_Battery_Plants")
assert len(battery)==200 and battery["plant_id"].is_unique
battery["county_geoid"] = county_key(battery)
if "nameplate_capacity_mw" not in battery:
    raise KeyError("I need EIA battery capacity_mw from Notebook 03 to estimate acreage")
battery["capacity_mw"] = pd.to_numeric(battery["nameplate_capacity_mw"],errors="coerce")
print("Missing or nonpositive MW:",int((battery["capacity_mw"].isna() | battery["capacity_mw"].le(0)).sum()))
assert battery["capacity_mw"].notna().all() and battery["capacity_mw"].gt(0).all(), "I must verify capacity coverage before statewide estimates"

Missing or nonpositive MW: 0


## Scenario definitions
The four scenario labels are `indiana_low`, `indiana_high`, `naco_low`, and `naco_high`. I additionally provide low, central, and high comparison columns using 0.03, 0.10, and 1.50 acres/MW, but the 0.10 central value is a **chosen comparison scenario**, not a statistically estimated median. Source leads from the original notebook: https://www.in.gov/oed/files/BESS-Handout-Siting-website.pdf and https://www.naco.org/resource/energy-technologies-and-siting-considerations-battery-energy-storage-systems . Verify definitions before publication.

In [3]:
rates={"indiana_low":0.03,"indiana_high":0.10,"naco_low":0.50,"naco_high":1.50}
assumptions=pd.DataFrame([{"scenario":s,"acres_per_mw":v,"basis":"Original Notebook 04 benchmark; verify site-area definition"} for s,v in rates.items()])
for name,rate in rates.items():battery[f"{name}_acres"]=battery["capacity_mw"]*rate
for scenario,rate in zip(SCENARIOS,[0.03,0.10,1.50]):battery[f"{scenario}_acres"]=battery["capacity_mw"]*rate
# I do not subtract whole battery plants based solely on solar co-location evidence.
state=pd.DataFrame([{"sector":"Battery","plants":len(battery),"capacity_mw":battery["capacity_mw"].sum(),**{f"{s}_acres":battery[f"{s}_acres"].sum() for s in SCENARIOS},**{f"{s}_acres":battery[f"{s}_acres"].sum() for s in rates}}])
county=battery.loc[battery["county_geoid"].notna()].groupby("county_geoid",as_index=False).agg(plants=("plant_id","nunique"),capacity_mw=("capacity_mw","sum"),**{f"{s}_acres":(f"{s}_acres","sum") for s in [*SCENARIOS,*rates]})
save_results(battery,county,state,assumptions)

Saved: G:\My Drive\Projects\Large projects\2026TexasEnergyFootprint\06_Energy_Footprint_Analysis\Battery


,sector,plants,capacity_mw,low_acres,central_acres,high_acres,indiana_low_acres,indiana_high_acres,naco_low_acres,naco_high_acres
0,Battery,200,13909.3,417.28,1390.93,20863.95,417.28,1390.93,6954.65,20863.95


,county_geoid,plants,capacity_mw,low_acres,central_acres,high_acres,indiana_low_acres,indiana_high_acres,naco_low_acres,naco_high_acres
0,48003,1,40.0,1.20,4.00,60.00,1.20,4.00,20.00,60.00
1,48005,2,19.8,0.59,1.98,29.70,0.59,1.98,9.90,29.70
2,48015,1,200.0,6.00,20.00,300.00,6.00,20.00,100.00,300.00
3,48025,1,9.9,0.30,0.99,14.85,0.30,0.99,4.95,14.85
4,48027,2,412.5,12.38,41.25,618.75,12.38,41.25,206.25,618.75
5,48029,3,69.9,2.10,6.99,104.85,2.10,6.99,34.95,104.85
6,48031,1,2.3,0.07,0.23,3.45,0.07,0.23,1.15,3.45
7,48033,1,150.0,4.50,15.00,225.00,4.50,15.00,75.00,225.00
8,48039,14,951.5,28.54,95.15,1427.25,28.54,95.15,475.75,1427.25
9,48043,1,73.2,2.20,7.32,109.80,2.20,7.32,36.60,109.80


In [4]:
# I calculate battery storage acreage under four benchmark rates
# and report the results for all 254 Texas counties.

from pathlib import Path
import pandas as pd
import numpy as np

OUTPUT = Path(OUTPUT)
OUTPUT.mkdir(parents=True, exist_ok=True)

# I preserve the four benchmark rates from the original analysis.
rates = {
    "indiana_low": 0.03,
    "indiana_high": 0.10,
    "naco_low": 0.50,
    "naco_high": 1.50
}

# I retain the three main scenarios for comparison across energy sectors.
scenario_rates = {
    "low": 0.03,
    "central": 0.10,
    "high": 1.50
}

assert set(SCENARIOS) == set(scenario_rates)

# I document each assumption and its preliminary source attribution.
assumptions = pd.DataFrame([
    {
        "scenario": name,
        "acres_per_mw": rate,
        "basis": "Original Notebook 04 benchmark; verify site-area definition"
    }
    for name, rate in rates.items()
])

# I verify the required input columns before calculating acreage.
required_columns = ["plant_id", "county_geoid", "capacity_mw"]

missing_columns = [
    col for col in required_columns
    if col not in battery.columns
]

assert not missing_columns, f"Missing columns: {missing_columns}"

# I check the capacity data because missing values would distort acreage estimates.
assert battery["capacity_mw"].notna().all(), (
    "Battery capacity contains missing values."
)
assert (battery["capacity_mw"] >= 0).all(), (
    "Battery capacity contains negative values."
)
assert battery["plant_id"].is_unique, (
    "Battery plant identifiers are not unique."
)

# I calculate acreage under each original benchmark.
for name, rate in rates.items():
    battery[f"{name}_acres"] = battery["capacity_mw"] * rate

# I calculate the three main acreage scenarios.
for scenario, rate in scenario_rates.items():
    battery[f"{scenario}_acres"] = battery["capacity_mw"] * rate

# I retain the gross battery estimate because co-location with solar
# does not establish the actual amount of overlapping land.
acreage_columns = [
    f"{name}_acres"
    for name in [*scenario_rates, *rates]
]

# I calculate the statewide battery inventory and acreage.
state = pd.DataFrame([{
    "sector": "Battery",
    "plants": len(battery),
    "capacity_mw": battery["capacity_mw"].sum(),
    **{
        col: battery[col].sum()
        for col in acreage_columns
    }
}])

# I standardize county identifiers before aggregation.
battery["county_geoid"] = (
    battery["county_geoid"]
    .astype("string")
    .str.strip()
    .str.zfill(5)
)

# I preserve missing county identifiers as missing rather than assigning them.
battery.loc[
    battery["county_geoid"].isin(["<NA>", "nan", "None", ""]),
    "county_geoid"
] = pd.NA

# I aggregate the battery inventory and acreage by county.
county = (
    battery.loc[battery["county_geoid"].notna()]
    .groupby("county_geoid", as_index=False)
    .agg(
        plants=("plant_id", "nunique"),
        capacity_mw=("capacity_mw", "sum"),
        **{
            col: (col, "sum")
            for col in acreage_columns
        }
    )
)

# I retrieve the official Texas county names and GEOIDs.
census_url = (
    "https://www2.census.gov/geo/docs/reference/codes2020/"
    "national_county2020.txt"
)

county_reference = pd.read_csv(
    census_url,
    sep="|",
    dtype=str,
    encoding="utf-8-sig"
)

county_reference = county_reference.loc[
    county_reference["STATEFP"] == "48",
    ["STATEFP", "COUNTYFP", "COUNTYNAME"]
].copy()

county_reference["county_geoid"] = (
    county_reference["STATEFP"]
    + county_reference["COUNTYFP"]
)

county_reference = county_reference.rename(
    columns={"COUNTYNAME": "county_name"}
)[["county_geoid", "county_name"]]

# I verify that the reference includes all Texas counties.
assert len(county_reference) == 254
assert county_reference["county_geoid"].is_unique

# I check that every assigned county matches an official Texas GEOID.
unrecognized = (
    set(county["county_geoid"])
    - set(county_reference["county_geoid"])
)

assert not unrecognized, (
    f"Unrecognized county GEOIDs: {sorted(unrecognized)}"
)

# I identify counties without battery plants in the dataset.
missing_counties = county_reference.loc[
    ~county_reference["county_geoid"].isin(county["county_geoid"])
].copy()

# I merge the battery calculations with all 254 Texas counties.
county_complete = county_reference.merge(
    county,
    on="county_geoid",
    how="left",
    validate="one_to_one"
)

# I assign zero to counties without battery records.
numeric_columns = [
    "plants",
    "capacity_mw",
    *acreage_columns
]

county_complete[numeric_columns] = (
    county_complete[numeric_columns].fillna(0)
)

county_complete["plants"] = (
    county_complete["plants"].astype(int)
)

# I sort the county table alphabetically for data checking.
county_complete = (
    county_complete
    .sort_values("county_name")
    .reset_index(drop=True)
)

# I verify that all 254 counties are represented.
assert len(county_complete) == 254
assert county_complete["county_geoid"].is_unique

# I check whether any battery plants lack county assignments.
unassigned = battery.loc[
    battery["county_geoid"].isna()
].copy()

print("Battery plants without county assignments:", len(unassigned))

# I reconcile statewide capacity and acreage with county-level totals.
# I report unassigned records separately rather than silently dropping them.
assert np.isclose(
    county_complete["capacity_mw"].sum()
    + unassigned["capacity_mw"].sum(),
    state["capacity_mw"].iloc[0],
    rtol=0,
    atol=0.01
)

for col in acreage_columns:
    assert np.isclose(
        county_complete[col].sum() + unassigned[col].sum(),
        state[col].iloc[0],
        rtol=0,
        atol=0.01
    ), f"County reconciliation failed for {col}"

# I save the facility-level, statewide, county-level, and assumptions outputs.
save_results(
    battery,
    county_complete,
    state,
    assumptions
)

# I save an explicitly named file containing all 254 Texas counties.
county_complete.to_csv(
    OUTPUT / "battery_county_254_with_names.csv",
    index=False
)

# I preserve any unassigned battery plants for the data checker.
if not unassigned.empty:
    unassigned.drop(
        columns="geometry",
        errors="ignore"
    ).to_csv(
        OUTPUT / "battery_unassigned_county_audit.csv",
        index=False
    )

# I report the final results.
print("\nFINAL BATTERY VALIDATION")
print("Total Texas counties:", len(county_complete))
print("Counties with battery plants:", (county_complete["plants"] > 0).sum())
print("Counties without battery plants:", (county_complete["plants"] == 0).sum())
print("Total battery plants:", len(battery))
print("Total capacity (MW):", f"{state['capacity_mw'].iloc[0]:,.2f}")

for scenario in SCENARIOS:
    print(
        f"{scenario.capitalize()} acreage:",
        f"{state[f'{scenario}_acres'].iloc[0]:,.2f}"
    )

print("\nPASS: Statewide acreage reconciles with county and unassigned totals.")

display(state)
display(county_complete)

Battery plants without county assignments: 0
Saved: G:\My Drive\Projects\Large projects\2026TexasEnergyFootprint\06_Energy_Footprint_Analysis\Battery


,sector,plants,capacity_mw,low_acres,central_acres,high_acres,indiana_low_acres,indiana_high_acres,naco_low_acres,naco_high_acres
0,Battery,200,13909.3,417.28,1390.93,20863.95,417.28,1390.93,6954.65,20863.95


,county_geoid,county_name,plants,capacity_mw,low_acres,central_acres,high_acres,indiana_low_acres,indiana_high_acres,naco_low_acres,naco_high_acres
0,48001,Anderson County,0,0.0,0.00,0.00,0.0,0.00,0.00,0.0,0.0
1,48003,Andrews County,1,40.0,1.20,4.00,60.0,1.20,4.00,20.0,60.0
2,48005,Angelina County,2,19.8,0.59,1.98,29.7,0.59,1.98,9.9,29.7
3,48007,Aransas County,0,0.0,0.00,0.00,0.0,0.00,0.00,0.0,0.0
4,48009,Archer County,0,0.0,0.00,0.00,0.0,0.00,0.00,0.0,0.0
5,48011,Armstrong County,0,0.0,0.00,0.00,0.0,0.00,0.00,0.0,0.0
6,48013,Atascosa County,0,0.0,0.00,0.00,0.0,0.00,0.00,0.0,0.0
7,48015,Austin County,1,200.0,6.00,20.00,300.0,6.00,20.00,100.0,300.0
8,48017,Bailey County,0,0.0,0.00,0.00,0.0,0.00,0.00,0.0,0.0
9,48019,Bandera County,0,0.0,0.00,0.00,0.0,0.00,0.00,0.0,0.0



FINAL BATTERY VALIDATION
Total Texas counties: 254
Counties with battery plants: 81
Counties without battery plants: 173
Total battery plants: 200
Total capacity (MW): 13,909.30
Low acreage: 417.28
Central acreage: 1,390.93
High acreage: 20,863.95

PASS: Statewide acreage reconciles with county and unassigned totals.


,sector,plants,capacity_mw,low_acres,central_acres,high_acres,indiana_low_acres,indiana_high_acres,naco_low_acres,naco_high_acres
0,Battery,200,13909.3,417.279,1390.93,20863.95,417.279,1390.93,6954.65,20863.95


,county_geoid,county_name,plants,capacity_mw,low_acres,central_acres,high_acres,indiana_low_acres,indiana_high_acres,naco_low_acres,naco_high_acres
0,48001,Anderson County,0,0.0,0.000,0.00,0.00,0.000,0.00,0.00,0.00
1,48003,Andrews County,1,40.0,1.200,4.00,60.00,1.200,4.00,20.00,60.00
2,48005,Angelina County,2,19.8,0.594,1.98,29.70,0.594,1.98,9.90,29.70
3,48007,Aransas County,0,0.0,0.000,0.00,0.00,0.000,0.00,0.00,0.00
4,48009,Archer County,0,0.0,0.000,0.00,0.00,0.000,0.00,0.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...
249,48499,Wood County,0,0.0,0.000,0.00,0.00,0.000,0.00,0.00,0.00
250,48501,Yoakum County,0,0.0,0.000,0.00,0.00,0.000,0.00,0.00,0.00
251,48503,Young County,1,9.9,0.297,0.99,14.85,0.297,0.99,4.95,14.85
252,48505,Zapata County,2,29.7,0.891,2.97,44.55,0.891,2.97,14.85,44.55
